In [7]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [1]:
# Helper functions
def add_user_message(messages, text) -> list[str]:
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text) -> list[str]:
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [9]:

# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [ ]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [ ]:
# helper functions for running the test cases through claude

from statistics import mean

TASK_PROMPT = """\
Please solve the following task:

{task}

* Respond only with Python, JSON, or a regex
* Do not add comments, commantary, or explanation
"""


def run_prompt(test_case) -> str:
    """Merges the prompt and test case input"""

    prompt = TASK_PROMPT.format(task=test_case["task"])
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code") #code is a bit more generic than we've seen before
    return chat(messages, stop_sequences=["```"])


def run_test_case(test_case) -> float:
    """Calls run_prompt and grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }


def run_eval(dataset):
    """Loads the dataset and runs run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average_score: {average_score}" )

    return results

In [10]:
import json

with open("dataset.json", "r") as file:
    dataset = json.load(file)

results = run_eval(dataset)

In [11]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS Account ID Extraction Function\n\n```python\ndef extract_account_id_from_arn(arn: str) -> str:\n    \"\"\"\n    Extracts the AWS account ID from an ARN (Amazon Resource Name) string.\n    \n    ARN format: arn:partition:service:region:account-id:resource-type/resource-id\n    \n    Args:\n        arn (str): The ARN string to parse\n        \n    Returns:\n        str: The AWS account ID (12-digit number), or empty string if not found\n        \n    Raises:\n        ValueError: If the ARN format is invalid\n        \n    Example:\n        >>> arn = \"arn:aws:iam::123456789012:user/username\"\n        >>> extract_account_id_from_arn(arn)\n        '123456789012'\n    \"\"\"\n    if not isinstance(arn, str):\n        raise ValueError(\"ARN must be a string\")\n    \n    if not arn.startswith(\"arn:\"):\n        raise ValueError(\"Invalid ARN format: must start with 'arn:'\")\n    \n    parts = arn.split(\":\")\n    \n    if len(parts) < 6:\n        raise ValueErr